# Performance by memorization

In [ ]:
import os
import sys
sys.path.append("..")
import pandas as pd
from src.Utils import read_jsonlines
from src.Eval import get_iobs_from_data, compute_results
from contextlib import redirect_stdout
import pandas as pd

output_path = "../data/intermediate/reddit+shsyt/"

def get_IOB_preds(output_path) -> pd.DataFrame:
    """Get multiindex dataframe with predicted IOB lists.
    Args:
        output_path (_type_): _description_
    Returns:
        pd.DataFrame: _description_
    """
    data = pd.DataFrame()

    for model in os.listdir(output_path):
        for f in os.listdir(os.path.join(output_path, model)):
            _ = f.split(".")[0].split("_")
            dataset, k = _[0], int(_[1].replace("shot", ""))
            sampling = _[2] if len(_) > 2 else ""

            preds = read_jsonlines(os.path.join(output_path, model, f))
            try: 
                _data = pd.DataFrame(preds).set_index("text")
                _, pred_iobs = get_iobs_from_data(preds)
            except:
                continue
            _data["IOB_pred"] = pred_iobs
            _data =  _data.loc[~_data.index.duplicated(keep="first"),["IOB_pred"]]
            
            _data.columns =  pd.MultiIndex.from_product([[model], [k], [sampling]])

            _new_cols = _data.columns.difference(data.columns)
    
            if not _new_cols.empty:
                data = pd.concat([data, _data[_new_cols]], axis=1, join='outer')
            else:
                data = data.combine_first(_data)
    
    return data
         
preds = get_IOB_preds(output_path)


In [ ]:


def get_model_pred_col(pred_df: pd.DataFrame, model: str) -> str:
    level_values = pred_df.columns.get_level_values(level=0)
    if model.lower() in level_values:
        return model.lower()
    elif model.replace("-", ":").lower() in level_values:
        return model.replace("-", ":").lower()
    return None

def suppress_output(func, *args, **kwargs):
    with open(os.devnull, 'w') as fnull:
        with redirect_stdout(fnull):
            return func(*args, **kwargs)      

data_memorization = pd.read_json("../output/memorization.jsonl", lines=True, orient="records")
data_joint = data_memorization
def get_results_per_factual_memorization_cls(k, sampling):
    
    correctness_cols = [c for c in data_joint.columns if "_Correctness" in c]
    results = {}   

    data_joint2 = data_joint.set_index(
        data_joint.TEXT.apply(' '.join)
    )

    for correctness_col in correctness_cols:
        model = correctness_col.split("_")[0]

        col = (get_model_pred_col(preds, model), k, sampling)

        results[model] = {}
        for group in data_joint[correctness_col].dropna().unique():
            try:
                _data = data_joint2.loc[data_joint2[correctness_col] == group].join(
                    preds.loc[:,col].rename("IOB_pred"), how="inner").dropna(subset=["IOB", "IOB_pred"])
                iob_true = _data.IOB.values
                iob_pred = _data.IOB_pred.values
                results[model][group] = {}

                kwargs = {"true_labels": iob_true, "true_predictions": iob_pred}
                values = suppress_output(compute_results, **kwargs)
                metrics = ["overall_strict_f1_macro", "overall_strict_recall_macro", "Artist_strict_recall", "WoA_strict_recall"]
                for metric in metrics:
                    results[model][group][metric] = values[metric]
                    results[model][group]["support"] = len(iob_true)
            except:
                print(f"Skipping for {col}")

    data = pd.json_normalize(results, sep='_')
    data.columns = pd.MultiIndex.from_tuples([tuple(col.split('_')) for col in data.columns])
    return data.stack(level=1)

k = 35
sampling = "tfidf"
get_results_per_factual_memorization_cls(k, sampling)


2024-12-05 17:28:58 root INFO: Imported 302 predictions for 302 true examples


2024-12-05 17:28:58 root INFO: Imported 389 predictions for 389 true examples
2024-12-05 17:28:58 root INFO: Imported 60 predictions for 60 true examples
2024-12-05 17:28:58 root INFO: Imported 420 predictions for 420 true examples
2024-12-05 17:28:58 root INFO: Imported 229 predictions for 229 true examples
2024-12-05 17:28:58 root INFO: Imported 103 predictions for 103 true examples
2024-12-05 17:28:58 root INFO: Imported 367 predictions for 367 true examples
2024-12-05 17:28:58 root INFO: Imported 332 predictions for 332 true examples
2024-12-05 17:28:58 root INFO: Imported 53 predictions for 53 true examples
2024-12-05 17:28:58 root INFO: Imported 388 predictions for 388 true examples
2024-12-05 17:28:58 root INFO: Imported 317 predictions for 317 true examples
2024-12-05 17:28:58 root INFO: Imported 46 predictions for 46 true examples


Skipping for (None, 35, 'tfidf')
Skipping for (None, 35, 'tfidf')
Skipping for (None, 35, 'tfidf')


FireFunction-v2                                       GPT-4o-mini  \
                  overall support   overall    Artist       WoA     overall   
                   strict     NaN    strict    strict    strict      strict   
                       f1     NaN    recall    recall    recall          f1   
                    macro     NaN     macro       NaN       NaN       macro   
0 Correct        0.778126     302  0.735976  0.784452  0.687500    0.862884   
  None           0.690476      60  0.650418  0.765625  0.535211    0.762484   
  Partial        0.804066     389  0.755007  0.833773  0.676240    0.824745   

                                                Llama3.1-70B          \
          support   overall    Artist       WoA      overall support   
              NaN    strict    strict    strict       strict     NaN   
              NaN    recall    recall    recall           f1     NaN   
              NaN     macro       NaN       NaN        macro     NaN   
0 Correct     229  0.852813  0.851852  0.853774     0.790389     367   
  None        103  0.721264  0.835052  0.607477     0.703902      53   
  Partial     420  0.794494  0.841346  0.747642     0.778282     332   

                                        Mixtral-8x22B                    \
            overall    Artist       WoA       overall support   overall   
             strict    strict    strict        strict     NaN    strict   
             recall    recall    recall            f1     NaN    recall   
              macro       NaN       NaN         macro     NaN     macro   
0 Correct  0.768175  0.752874  0.783476      0.830619     388  0.796397   
  None     0.659251  0.708333  0.610169      0.754167      46  0.713333   
  Partial  0.746246  0.765766  0.726727      0.797246     317  0.762025   

                               
             Artist       WoA  
             strict    strict  
             recall    recall  
                NaN       NaN  
0 Correct  0.784574  0.808219  
  None     0.760000  0.666667  
  Partial  0.840000  0.684049